# Prep ORPO Training Data

Phase 1 backtracking augmentation: for each assistant turn in `all_messages_cleaned.json`,
calls GPT-4o-mini with the backtracking prompt to infer what `(analysis, angle, draft)`
would most naturally have led to the actual reply.

Output: `output/augmented_phase1.csv` — one row per assistant turn,
containing everything needed to later run Phase 2 fine-tuning.

In [17]:
import csv
import json
import logging
import os
import sys
import time
from pathlib import Path

import requests

# Add project root to path AND set cwd so pydantic-settings finds .env
try:
    _ROOT = Path(__vsc_ipynb_file__).parent.parent  # chatbot/
except NameError:
    _ROOT = Path.cwd()

os.chdir(_ROOT)  # ensures settings picks up .env from project root

if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from api.config import settings

DATA_PATH             = _ROOT / "output" / "all_messages_cleaned.json"
OUT_CSV               = _ROOT / "output" / "augmented_phase1.csv"
BACKTRACK_PROMPT_PATH = _ROOT / "training" / "prompts" / "brainstorm_backtrack_v1.txt"

CSV_COLUMNS = [
    "conversation_id",
    "turn_index",
    "context_formatted",
    "last_user_message",
    "phase1_keywords",   # JSON string: [{"word": ..., "significance": ...}]
    "phase1_analysis",
    "phase1_angle",
    "phase1_draft",
    "phase1_raw",
    "expected_reply",
    "summary",
]

API_DELAY_SECONDS = 0.5

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

print("Project root:", _ROOT)
print("Data path:   ", DATA_PATH)
print("Output CSV:  ", OUT_CSV)
print("Using model: ", settings.judge_model)
print("API key set: ", bool(settings.openrouter_api_key))

Project root: /Users/school/Ian's Chatbot/chatbot
Data path:    /Users/school/Ian's Chatbot/chatbot/output/all_messages_cleaned.json
Output CSV:   /Users/school/Ian's Chatbot/chatbot/output/augmented_phase1.csv
Using model:  openai/gpt-4o-mini
API key set:  True


In [18]:
# ── Load data + backtracking prompt ──────────────────────────────────────────
with open(DATA_PATH, "r", encoding="utf-8") as f:
    conversations = json.load(f)

with open(BACKTRACK_PROMPT_PATH, "r", encoding="utf-8") as f:
    backtrack_template = f.read().strip()

total_messages = sum(len(v) for v in conversations.values())
total_assistant = sum(
    sum(1 for m in v if m["role"] == "assistant")
    for v in conversations.values()
)

print(f"Loaded {len(conversations)} conversations, {total_messages} total messages")
print(f"Assistant turns (rows to augment): {total_assistant}")

Loaded 73 conversations, 732 total messages
Assistant turns (rows to augment): 383


In [19]:
# ── Helper functions ──────────────────────────────────────────────────────────

def format_conversation(messages: list[dict]) -> str:
    """Convert messages list to Friend:/You: format string."""
    lines = []
    for msg in messages:
        prefix = "Friend:" if msg["role"] == "user" else "You:"
        content_lines = msg["content"].split("\n")
        for i, cl in enumerate(content_lines):
            lines.append(f"{prefix} {cl}" if i == 0 else f"  {cl}")
    return "\n".join(lines)


def build_backtrack_messages(
    template: str,
    context: list[dict],
    actual_reply: str,
) -> list[dict]:
    """Fill {CONVERSATION} and {ACTUAL_REPLY} into the backtracking prompt."""
    conversation_text = format_conversation(context) if context else "（对话开头，没有前文）"
    system_content = (
        template
        .replace("{CONVERSATION}", conversation_text)
        .replace("{ACTUAL_REPLY}", actual_reply)
    )
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": "请分析对话，输出JSON。"},
    ]


def call_phase1_sync(messages: list[dict]) -> str:
    """Synchronous requests call to OpenRouter. Returns raw response content string."""
    headers = {
        "Authorization": f"Bearer {settings.openrouter_api_key}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": settings.judge_model,
        "messages": messages,
        "temperature": 0.7,
    }
    resp = requests.post(
        f"{settings.base_url}/chat/completions",
        headers=headers,
        json=payload,
        timeout=60,
    )
    resp.raise_for_status()
    data = resp.json()
    try:
        return data["choices"][0]["message"]["content"]
    except (KeyError, IndexError) as e:
        raise ValueError(f"Unexpected LLM response shape: {data}") from e


def parse_phase1_response(raw: str) -> tuple[str, str, str, str]:
    """
    Extract (keywords_json, analysis, angle, draft).
    keywords_json is a JSON string of the keywords list.
    Returns ('', '', '', '') on any failure.
    """
    clean = raw.strip()
    if clean.startswith("```"):
        parts = clean.split("```")
        inner = parts[1]
        if inner.startswith("json"):
            inner = inner[4:]
        clean = inner.strip()
    try:
        data = json.loads(clean)
        keywords_json = json.dumps(data.get("keywords", []), ensure_ascii=False)
        return (
            keywords_json,
            data.get("analysis", ""),
            data.get("angle", ""),
            data.get("draft", ""),
        )
    except json.JSONDecodeError:
        return "", "", "", ""


def iter_assistant_turns(
    messages: list[dict],
) -> list[tuple[int, list[dict], str, str]]:
    """
    Return (turn_index, context, last_user_message, expected_reply)
    for every assistant turn in the conversation.
    """
    results = []
    turn_idx = 0
    for i, msg in enumerate(messages):
        if msg["role"] != "assistant":
            continue
        context = messages[:i]
        last_user = ""
        for m in reversed(context):
            if m["role"] == "user":
                last_user = m["content"]
                break
        results.append((turn_idx, context, last_user, msg["content"]))
        turn_idx += 1
    return results


def load_already_processed(csv_path) -> set:
    """Return set of (conversation_id, turn_index) already in CSV."""
    done = set()
    if not Path(csv_path).exists():
        return done
    with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            try:
                done.add((row["conversation_id"], int(row["turn_index"])))
            except (KeyError, ValueError):
                pass
    return done


print("Helper functions defined.")

Helper functions defined.


In [20]:
# ── SAMPLE TEST — run all turns in first 3 conversations ────────────────────
# Adjust SAMPLE_CONVOS to test different conversations.

SAMPLE_CONVOS = list(sorted(conversations.keys()))[:3]

for convo_id in SAMPLE_CONVOS:
    messages = conversations[convo_id]
    turns = iter_assistant_turns(messages)
    if not turns:
        print(f"\n[{convo_id}] No assistant turns — skipping")
        continue

    print(f"\n{'='*60}")
    print(f"Conversation: {convo_id}  |  {len(turns)} assistant turn(s)")

    for turn_idx, context, last_user_msg, expected_reply in turns:
        print(f"\n  --- Turn {turn_idx} ---")
        print(f"  Context ({len(context)} messages):")
        print(format_conversation(context) if context else "  (none)")
        print(f"\n  Expected reply (shown to backtrack model):\n    {expected_reply!r}")

        msgs = build_backtrack_messages(backtrack_template, context, expected_reply)
        raw = call_phase1_sync(msgs)
        keywords_json, analysis, angle, draft = parse_phase1_response(raw)

        keywords = json.loads(keywords_json) if keywords_json else []
        print(f"\n  Phase 1 backtracked output:")
        for kw in keywords:
            print(f"    keyword : {kw.get('word')}  →  {kw.get('significance')}")
        print(f"    analysis: {analysis}")
        print(f"    angle   : {angle}")
        print(f"    draft   : {draft}")
        time.sleep(API_DELAY_SECONDS)

print("\nSample test done. Review outputs above before running the full pipeline.")


Conversation: convo1  |  1 assistant turn(s)

  --- Turn 0 ---
  Context (1 messages):
Friend: 呐
  把好天气分享给你

  Expected reply (shown to backtrack model):
    '哇哇哇\n是谁有这么好的运气能被分享好天气！\n是谁！\n原来是我这个幸运的人'

  Phase 1 backtracked output:
    keyword : 好天气  →  字面意思是天气很好，潜台词暗示了分享快乐的心情。
    analysis: 对方分享好天气，情绪积极，表达了一种愉悦的心情，因此顺着这个方向回应能加强互动的轻松感。选这个方向是为了共鸣对方的快乐。
    angle   : 用调侃的方式回应‘好天气’的分享，增强互动感。
    draft   : 哇，真是太好了！能被分享这么好的天气，感觉自己真的很幸运！你有没有什么好玩的计划？

Conversation: convo11  |  4 assistant turn(s)

  --- Turn 0 ---
  Context (1 messages):
Friend: 被采购气死了
  就是上次和你说的那个
  无语
  沟通太费劲了
  厌蠢症犯了

  Expected reply (shown to backtrack model):
    '又犯蠢了？\n你直接开骂\n不要忍着\n给你点杯苦瓜柠檬茶消消火?'

  Phase 1 backtracked output:
    keyword : 气死  →  字面意思是非常生气，潜台词是对采购的无奈和不满。
    keyword : 沟通太费劲了  →  字面意思是沟通困难，潜台词是感到疲惫和挫败。
    keyword : 厌蠢症  →  字面意思是对无意义的事情感到厌烦，潜台词是想要发泄对现状的不满。
    analysis: Friend的情绪很消极，表达了对采购和沟通的强烈不满。选择调侃的方式回应，能够在轻松氛围中让对方释放情绪。并且通过提供饮品的建议，体现了关心和支持。
    angle   : 通过调侃'又犯蠢了'和提供饮品的建议，轻松应对Friend的抱怨。
    draft

In [ ]:
# ── FULL RUN (resumable) ──────────────────────────────────────────────────────
# Safe to re-run — already-processed (convo_id, turn_index) pairs are skipped.

already_done = load_already_processed(OUT_CSV)
append_mode  = bool(already_done)

csv_file = open(OUT_CSV, "a" if append_mode else "w", encoding="utf-8-sig", newline="")
writer   = csv.DictWriter(csv_file, fieldnames=CSV_COLUMNS)
if not append_mode:
    writer.writeheader()

total_turns = processed = skipped = errors = 0

try:
    for convo_id, messages in sorted(conversations.items()):
        turns = iter_assistant_turns(messages)
        if not turns:
            continue

        for turn_idx, context, last_user_msg, expected_reply in turns:
            total_turns += 1

            if (convo_id, turn_idx) in already_done:
                skipped += 1
                continue

            context_formatted = format_conversation(context) if context else ""
            msgs = build_backtrack_messages(backtrack_template, context, expected_reply)

            phase1_raw = phase1_keywords = phase1_analysis = phase1_angle = phase1_draft = ""

            try:
                phase1_raw = call_phase1_sync(msgs)
                phase1_keywords, phase1_analysis, phase1_angle, phase1_draft = parse_phase1_response(phase1_raw)
                if not phase1_angle:
                    logger.warning(
                        "Unparseable JSON for %s turn %d: %r",
                        convo_id, turn_idx, phase1_raw[:200],
                    )
                    errors += 1
                else:
                    processed += 1
            except requests.HTTPError as exc:
                logger.warning("HTTP error %s turn %d: %s", convo_id, turn_idx, exc)
                errors += 1
            except Exception as exc:
                logger.warning("Error %s turn %d: %s", convo_id, turn_idx, exc)
                errors += 1

            writer.writerow({
                "conversation_id":   convo_id,
                "turn_index":        turn_idx,
                "context_formatted": context_formatted,
                "last_user_message": last_user_msg,
                "phase1_keywords":   phase1_keywords,
                "phase1_analysis":   phase1_analysis,
                "phase1_angle":      phase1_angle,
                "phase1_draft":      phase1_draft,
                "phase1_raw":        phase1_raw,
                "expected_reply":    expected_reply,
                "summary":           "",
            })
            csv_file.flush()
            time.sleep(API_DELAY_SECONDS)

finally:
    csv_file.close()

print(f"\nDone.")
print(f"  Total turns : {total_turns}")
print(f"  Skipped     : {skipped}")
print(f"  Processed   : {processed}")
print(f"  Errors      : {errors}")
print(f"  Output      : {OUT_CSV}")

In [ ]:
# ── Inspect output ─────────────────────────────────────────────────────────────
import pandas as pd

df = pd.read_csv(OUT_CSV)
print(f"CSV rows: {len(df)}  |  Columns: {df.columns.tolist()}")
print(f"Conversations covered: {df['conversation_id'].nunique()}")
print(f"Rows with valid phase1_angle: {df['phase1_angle'].notna().sum()} / {len(df)}")

# Show a sample of key columns
pd.set_option("display.max_colwidth", 80)
df[["conversation_id", "turn_index", "phase1_analysis", "phase1_angle", "phase1_draft", "expected_reply"]].head(10)